# Strategy v5 on Colab — GPU stages (A0–A4), optional CPU stages (A5–B)

Runs the bi-encoder part of `strategy_v5.md` on a Colab GPU and leaves every output on Google Drive so
the CPU part (union → GBDT → decision → submission) can run on the laptop with
`.\run_v5.ps1 -CpuOnly`, or here with `RUN_CPU_STAGES = True` (slow on Colab's 2 cores, needs the high-RAM runtime).

**Before you start — put the inputs on Drive** (one of the two):

* a folder `MyDrive/AWSomeMinds/inputs/` (recommended: upload the folder with the Drive web UI or Drive for desktop),
  **or** `MyDrive/AWSomeMinds/inputs.zip` (note: PowerShell's `Compress-Archive` fails on files > 2 GB; use 7-Zip), containing
  ```
  artifacts/norm/{train,test}_source{1,2,3}.parquet          normalized text (6 files)
  artifacts/baseline/train/{India,US}.parquet + *_entities.parquet
  artifacts/baseline/test/{France,India,US}.parquet + *_entities.parquet
  artifacts/folds.parquet
  artifacts/neural/train_pairs.parquet, eval_records.parquet   A1 output from the laptop (else add data/dataset/train/train_ground_truth.tsv)
  data/dataset/                                                only if RUN_CPU_STAGES (predict needs the test TSVs)
  ```
  Build that folder on the laptop from the repo folder (PowerShell), then upload `inputs\` to Drive (~7.5 GB):
  ```powershell
  New-Item -ItemType Directory -Force inputs\artifacts | Out-Null
  Copy-Item artifacts\norm, artifacts\baseline, artifacts\neural -Destination inputs\artifacts -Recurse
  Copy-Item artifacts\folds.parquet inputs\artifacts
  ```

**Outputs persist under `MyDrive/AWSomeMinds/artifacts/{neural,dense}`** (training checkpoints, `eval.json`,
dense candidates). If Colab disconnects: reconnect, **Runtime ▸ Run all**. Finished stages are skipped,
training resumes from its last checkpoint, per-source embeddings are reused within the session.

**Runtime ▸ Change runtime type ▸ GPU.** Rough budget with `multilingual-e5-base` on all 3.06M pairs:
A100/L4 ≈ 3–5 h total; T4 ≈ 9–10 h (use `e5-small` or `N_PAIRS = 1500000` there). Keep the tab open.


In [ ]:
#@title 1. Parameters
DRIVE_DIR = "/content/drive/MyDrive/AWSomeMinds"  #@param {type:"string"}
REPO_URL = "https://github.com/anujkushwaha612/AWSomeMinds.git"  #@param {type:"string"}
BRANCH = "arena/01a0da82-awsomeminds"  #@param {type:"string"}
MODEL = "auto"  #@param ["auto", "intfloat/multilingual-e5-base", "intfloat/multilingual-e5-small"]
N_PAIRS = 0  #@param {type:"integer"}
RUN_CPU_STAGES = False  #@param {type:"boolean"}
# MODEL "auto": e5-base on >= 20 GB VRAM (L4 / A100 / 48 GB), e5-small on a T4.
# N_PAIRS 0 = all folds-3/4 positives (3.06M); e.g. 1500000 halves training time.


In [ ]:
#@title 2. Mount Drive, clone the repo, install
from google.colab import drive
drive.mount('/content/drive')

import os, sys, subprocess, pathlib, shutil, json, time
WORK = "/content/AWSomeMinds"
if not os.path.isdir(WORK):
    subprocess.run(["git", "clone", "-q", "--branch", BRANCH, REPO_URL, WORK], check=True)
os.chdir(WORK)
subprocess.run(["git", "pull", "-q", "--ff-only"], check=False)
print("commit:", subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip())
# torch / numpy / pandas / pyarrow stay as Colab ships them (matching CUDA + ABI builds)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", "."], check=True)
os.environ["PYTHONPATH"] = f"{WORK}/src"
os.environ["PYTHONIOENCODING"] = "utf-8"
os.environ["PYTHONWARNINGS"] = "ignore"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

def run(*module_args, check=True):
    """Run `python -u -m <module> ...` in the repo, streaming its output; raise if it fails."""
    cmd = [sys.executable, "-u", "-m", *module_args]
    print(">>>", " ".join(cmd), time.strftime("%H:%M:%S"), flush=True)
    p = subprocess.Popen(cmd, cwd=WORK, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=os.environ)
    for line in p.stdout:
        print(line, end="", flush=True)
    code = p.wait()
    if check and code != 0:
        raise RuntimeError(f"stage failed (exit {code}) - fix or simply re-run this cell to resume")
    return code
print("ok")


In [ ]:
#@title 3. Stage the inputs (Drive -> local disk) and link the outputs to Drive
DRIVE = pathlib.Path(DRIVE_DIR)
ART = pathlib.Path(WORK, "artifacts"); ART.mkdir(exist_ok=True)
zip_path, folder = DRIVE / "inputs.zip", DRIVE / "inputs"
if not (ART / "norm").exists():                       # first run in this session
    t = time.time()
    if zip_path.exists():
        print("unzipping", zip_path)
        subprocess.run(["unzip", "-q", "-o", str(zip_path), "-d", WORK], check=True)
    elif folder.exists():
        print("copying", folder)
        subprocess.run(["rsync", "-a", f"{folder}/", f"{WORK}/"], check=True)
    else:
        raise FileNotFoundError(f"put inputs.zip or an inputs/ folder in {DRIVE} (see the first cell)")
    print(f"inputs staged in {time.time() - t:.0f}s")
    # tolerate any layout: find the norm/ folder wherever it landed and move its siblings under artifacts/
    if not (ART / "norm").exists():
        hits = [h for h in pathlib.Path(WORK).glob("**/norm/train_source1.parquet") if ".git" not in h.parts]
        if not hits:
            raise FileNotFoundError("no norm/train_source1.parquet inside the inputs")
        src_root = hits[0].parent.parent                # the folder that plays the role of artifacts/
        for item in list(src_root.iterdir()):
            if item.name in ("norm", "baseline", "neural", "folds.parquet", "cache") and not (ART / item.name).exists():
                shutil.move(str(item), str(ART / item.name))
        data_hits = [h for h in pathlib.Path(WORK).glob("**/data/dataset/test/test_source1.tsv")]
        if data_hits and not pathlib.Path(WORK, "data/dataset").exists():
            shutil.move(str(data_hits[0].parent.parent), str(pathlib.Path(WORK, "data/dataset")))

# outputs that must survive a disconnect live on Drive; the repo folder just links to them
for name in ("neural", "dense") + (("union", "v5") if RUN_CPU_STAGES else ()):
    target = DRIVE / "artifacts" / name
    target.mkdir(parents=True, exist_ok=True)
    link = ART / name
    if link.is_symlink():
        continue
    if link.exists():                                 # e.g. A1 files that came with the inputs
        for f in link.iterdir():
            if not (target / f.name).exists():
                shutil.move(str(f), str(target / f.name))
        shutil.rmtree(link)
    link.symlink_to(target)
if RUN_CPU_STAGES:
    for name in ("output", "subs"):
        target = DRIVE / name; target.mkdir(parents=True, exist_ok=True)
        link = pathlib.Path(WORK, name)
        if link.exists() and not link.is_symlink():
            shutil.rmtree(link)
        if not link.exists():
            link.symlink_to(target)

need = ["norm/train_source1.parquet", "norm/test_source3.parquet", "baseline/train/US.parquet",
        "baseline/train/US_entities.parquet", "baseline/test/France.parquet"]
missing = [f for f in need if not (ART / f).exists()]
if missing:
    raise FileNotFoundError(f"missing inputs: {missing}")
a1 = all((ART / "neural" / f).exists() for f in ("train_pairs.parquet", "eval_records.parquet"))
print("A1 pairs present:", a1, "| free disk:", subprocess.run(["df", "-h", "/content"], capture_output=True, text=True).stdout.split("\n")[1])


In [ ]:
#@title 4. GPU profile -> config override (nothing in the repo is edited)
import torch, yaml
cfg = yaml.safe_load(open("configs/pipeline.yaml", encoding="utf-8"))
n = cfg["v5"]["neural"]
if not torch.cuda.is_available():
    raise RuntimeError("no GPU: Runtime > Change runtime type > GPU")
prop = torch.cuda.get_device_properties(0)
vram = prop.total_memory / 2**30
if vram >= 40:      # A100 40/80 GB, the 48 GB box: everything resident
    profile = dict(grad_checkpointing=False, emb_store="gpu", tile_gb=6.0, encode_batch=512, batch_size=256)
elif vram >= 20:    # L4 24 GB, A10 24 GB
    profile = dict(grad_checkpointing=True, emb_store="disk", tile_gb=3.0, encode_batch=512, batch_size=256)
else:               # T4 16 GB: also checkpoint more often (free sessions are fragile)
    profile = dict(grad_checkpointing=True, emb_store="disk", tile_gb=1.5, encode_batch=256, batch_size=256, ckpt_every=1000)
n.update(profile)
n["emb_dir"] = "/content/emb"                        # fast local disk; memmaps are per-session scratch
n["model"] = MODEL if MODEL != "auto" else ("intfloat/multilingual-e5-base" if vram >= 20 else "intfloat/multilingual-e5-small")
n["n_pairs"] = int(N_PAIRS)
import psutil
ram = psutil.virtual_memory().total / 2**30
if RUN_CPU_STAGES and ram < 24:
    cfg["v5"]["max_train_rows"] = 6_000_000           # stage-1 matrix cap for a ~12 GB runtime
CONFIG = "/content/colab_config.yaml"
yaml.safe_dump(cfg, open(CONFIG, "w", encoding="utf-8"))
os.environ["BER_CONFIG"] = CONFIG
print(f"GPU {prop.name}  VRAM {vram:.1f} GB  bf16 {torch.cuda.is_bf16_supported()}  RAM {ram:.1f} GB")
print("neural config:", {k: n[k] for k in ("model", "n_pairs", "batch_size", "grad_checkpointing", "emb_store", "tile_gb", "encode_batch", "max_len")})


In [ ]:
#@title 5. Tests (~30 s) and A0 throughput probe (~5 min)
run("pytest", "-q", "-x")
run("ber.neural.env_check")


In [ ]:
#@title 6. A1 training pairs (skipped if they came from the laptop)
if a1 or all((ART / "neural" / f).exists() for f in ("train_pairs.parquet", "eval_records.parquet")):
    print("A1 output present, skipping")
else:
    run("ber.neural.pairs")        # needs data/dataset/train/train_ground_truth.tsv in the inputs


In [ ]:
#@title 7. A2-A3 fine-tune the bi-encoder + recall gate (the long one; resumable)
# If the session dies mid-way: reconnect, run cells 1-4, then this cell again. It resumes from the
# last checkpoint (ckpt_every steps) on Drive. If you CHANGE the model or N_PAIRS afterwards, delete
# MyDrive/AWSomeMinds/artifacts/neural/biencoder first (the code refuses to resume a mismatching checkpoint).
state = ART / "neural" / "biencoder" / "train_state.json"
if state.exists():
    s = json.load(open(state))
    print("checkpoint:", s)
    if s["step"] >= s["total"] and (ART / "neural" / "eval.json").exists():
        print("training and gate already done, skipping"); done = True
    else:
        done = False
else:
    done = False
if not done:
    run("ber.neural.train_biencoder")
print(json.dumps(json.load(open(ART / "neural" / "eval.json")), indent=1))


In [ ]:
#@title 8. A4 dense search: train, then test (resumable per country)
run("ber.neural.dense_retrieve", "--split", "train")
shutil.rmtree("/content/emb/train", ignore_errors=True)   # scratch; the results are on Drive
run("ber.neural.dense_retrieve", "--split", "test")
shutil.rmtree("/content/emb/test", ignore_errors=True)
for split in ("train", "test"):
    for f in sorted((ART / "dense" / split).glob("*")):
        print(f"{split}/{f.name:32s} {f.stat().st_size / 2**20:8.1f} MB")


In [ ]:
#@title 9. Summary: retrieval gate and what to do next
ev = json.load(open(ART / "neural" / "eval.json"))
print(f"encoder: {ev['encoder']}")
print(f"{'country':8s} {'group':6s} {'n':>7s} {'tfidf R@3':>10s} {'dense R@3':>10s} {'dense R@5':>10s} {'union R@3+3':>12s}")
for country, groups in ev.items():
    if not isinstance(groups, dict): continue
    for g, r in groups.items():
        print(f"{country:8s} {g:6s} {r['n']:7d} {r['tfidf_R@3']:10.4f} {r['dense_R@3']:10.4f} {r['dense_R@5']:10.4f} {r['union_R@3+3']:12.4f}")
print("GATE dense R@3 >= TF-IDF R@3 in every country:", ev["gate_dense_R@3_ge_tfidf_R@3"])
print("""
The fold-0 macro F0.5 (the number to compare with the baseline's 0.9487 offline / 0.938 LB) comes out of
the CPU stages. Either:
  * laptop:  download MyDrive/AWSomeMinds/artifacts/{neural,dense} into the repo's artifacts\ folder, then
             powershell -ExecutionPolicy Bypass -File .\run_v5.ps1 -CpuOnly
             -> artifacts\v5\stage2.json (fold-0 F0.5), compare.json (vs baseline), output\matching_results.tsv
  * here:    set RUN_CPU_STAGES = True in cell 1 and run the next cell (slow on 2 cores; high-RAM runtime).
""")


In [ ]:
#@title 10. (optional) CPU stages on Colab: union -> stage 1 -> stage 2 -> compare -> predict
if not RUN_CPU_STAGES:
    print("RUN_CPU_STAGES is False - skipping")
else:
    run("ber.union", "--split", "train")
    run("ber.union", "--split", "test")
    run("ber.v5", "stage1")
    run("ber.v5", "stage2")
    run("ber.v5", "compare")
    for f in ("stage1.json", "stage2.json", "compare.json"):
        p = ART / "v5" / f
        if p.exists():
            print(f, json.dumps(json.load(open(p)), indent=1)[:3000])
    if pathlib.Path(WORK, "data/dataset/test/test_source1.tsv").exists():
        run("ber.v5", "predict", "--name", "sub03_v5_colab")   # writes output/ and subs/ on Drive
        run("ber.gap", "stress", "--run", "v5", check=False)
    else:
        print("data/dataset/test missing: predict skipped (metrics above are complete without it)")
